# ID3 from scratch

Entropy + information gain; recursive categorical decision tree.

**No scikit-learn is used.** Requirements: `pandas`, `openpyxl`, `numpy`, `matplotlib`. Put `data.xlsx` beside the notebook. Change `TARGET` if the last column is not your class label.

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from math import log2

FILE = "data.xlsx"
df = pd.read_excel(FILE).dropna().copy()
print(df.head())
print("Columns:", list(df.columns))

TARGET = df.columns[-1]  # change this if your target column is different

def entropy(y):
    counts = Counter(y)
    n = len(y)
    return -sum((v/n) * log2(v/n) for v in counts.values() if v)

def info_gain(data, col, target):
    base = entropy(data[target])
    n = len(data)
    rem = sum(len(g)/n * entropy(g[target]) for _, g in data.groupby(col))
    return base - rem

def id3(data, features, target):
    if data[target].nunique() == 1:
        return data[target].iloc[0]
    if not features:
        return data[target].mode().iloc[0]
    gains = {f: info_gain(data, f, target) for f in features}
    best = max(gains, key=gains.get)
    node = {"feature": best, "branches": {}}
    for value, group in data.groupby(best):
        node["branches"][value] = id3(group, [f for f in features if f != best], target)
    return node

def predict_one(tree, row, fallback):
    if not isinstance(tree, dict):
        return tree
    value = row[tree["feature"]]
    branch = tree["branches"].get(value)
    return fallback if branch is None else predict_one(branch, row, fallback)

features = [c for c in df.columns if c != TARGET]
tree = id3(df, features, TARGET)
fallback = df[TARGET].mode().iloc[0]
pred = df.apply(lambda r: predict_one(tree, r, fallback), axis=1)
print("Tree:", tree)
print("Training accuracy:", round((pred == df[TARGET]).mean(), 3))
pd.Series(pred).value_counts().plot(kind="bar", title="ID3 predicted class counts")
plt.xlabel("Predicted class"); plt.ylabel("Count"); plt.tight_layout(); plt.show()
